# Deep learning on images

## Load modules from repo

In [150]:
# Le code suivant dans un notebook permet de :
# - autoriser les imports de fichiers python de ce repo
# - spécifier les chemins relativement à la racine du repo plutôt que relativement au notebook

import os

# Ce code cherche le dossier racine en remontant dans l'arborescence
# jusqu'à ce qu'il trouve le dossier 'src'.
# Cela le rend indépendant de l'endroit où vous lancez le notebook.
try:
    # On part du dossier du notebook
    notebook_dir = os.path.dirname(__file__)
except NameError:
    # __file__ n'existe pas en mode interactif, on utilise le répertoire de travail
    notebook_dir = os.getcwd()

# On remonte jusqu'à trouver un dossier contenant 'src'
project_root = notebook_dir
while not os.path.isdir(os.path.join(project_root, 'src')):
    parent_dir = os.path.dirname(project_root)
    if parent_dir == project_root: # On a atteint la racine du système
        raise FileNotFoundError("Impossible de trouver le dossier 'src'. Vérifiez la structure du projet.")
    project_root = parent_dir

os.chdir(project_root)

In [151]:
os.getcwd()

'/home/val/Documents/Dev/DataScientest/Rakuten'

In [152]:
import src
from src.preprocessing.core import load_reproducible_split
from src.preprocessing.image import get_image_path, load_image, get_image_features_with_hash, get_image_md5_hash

In [153]:
import importlib
importlib.reload(src.preprocessing.core)
importlib.reload(src.preprocessing.image)

<module 'src.preprocessing.image' from '/home/val/Documents/Dev/DataScientest/Rakuten/src/preprocessing/image.py'>

## Preprocessing

In [154]:
from pathlib import Path
artifacts_folder=Path('artifacts/on_images/deep_learning/v1')

In [155]:
create_and_save_artifacts=True  # False to simply load artifacts

### Load split dataset

In [156]:
X_train, X_test, y_train, y_test = load_reproducible_split(folder = 'Dataset2')

In [157]:
X_train['image_path']=X_train.apply(get_image_path,axis=1,as_string=True)
X_test['image_path']=X_test.apply(get_image_path,axis=1,as_string=True)

In [159]:
columns_to_drop=['designation', 'description', 'gray_image_pHash', 'productid', 'imageid']
X_train = X_train.drop(columns=columns_to_drop)
X_test = X_test.drop(columns=columns_to_drop)

In [160]:
X_train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 67932 entries, 1887 to 10092
Data columns (total 27 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   image_pHash             67932 non-null  object 
 1   mean_r                  67932 non-null  float64
 2   mean_g                  67932 non-null  float64
 3   mean_b                  67932 non-null  float64
 4   std_r                   67932 non-null  float64
 5   std_g                   67932 non-null  float64
 6   std_b                   67932 non-null  float64
 7   median_r                67932 non-null  float64
 8   median_g                67932 non-null  float64
 9   median_b                67932 non-null  float64
 10  mean_gray               67932 non-null  float64
 11  std_gray                67932 non-null  float64
 12  median_gray             67932 non-null  float64
 13  essential_pixel_count   67932 non-null  int64  
 14  x_min                   67932 non-null  

### Encoding of target

#### LabelEncoder

Using a LabelEncoder is needed by to_categorical.

In [161]:
y_train.info()

<class 'pandas.core.series.Series'>
Index: 67932 entries, 1887 to 10092
Series name: prdtypecode
Non-Null Count  Dtype
--------------  -----
67932 non-null  int64
dtypes: int64(1)
memory usage: 1.0 MB


In [162]:
import joblib
path=artifacts_folder / 'y_label_encoder.joblib'
path

PosixPath('artifacts/on_images/deep_learning/v1/y_label_encoder.joblib')

In [163]:
from sklearn.preprocessing import LabelEncoder
if create_and_save_artifacts:
    label_encoder = LabelEncoder()
    label_encoder.fit(y_train)

    # Save
    path.parents[0].mkdir(parents=True, exist_ok=True)
    joblib.dump(label_encoder, path)

In [164]:
# Load
label_encoder = joblib.load(path)

In [165]:
num_classes=len(label_encoder.classes_)
num_classes

27

In [166]:
y_train = label_encoder.transform(y_train)
y_test = label_encoder.transform(y_test)

#### One-hot encoding

In [167]:
from tensorflow.keras.utils import to_categorical
y_train = to_categorical(y_train, num_classes=num_classes)
y_test = to_categorical(y_test, num_classes=num_classes)

### Encoding of features


#### Numeric

In [168]:
numeric_features=['mean_r', 'mean_g', 'mean_b', 'std_r', 'std_g', 'std_b', 'median_r', 'median_g', 'median_b', 'mean_gray', 'std_gray', 'median_gray', 'essential_pixel_count', 'x_min', 'y_min', 'x_max', 'y_max', 'len_designation', 'len_description', 'essential_width', 'essential_height', 'essential_aspect_ratio', 'essential_area', 'rectangleness']

In [169]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler

In [170]:
import joblib
path=artifacts_folder / 'tabular_preprocessor.joblib'
path

PosixPath('artifacts/on_images/deep_learning/v1/tabular_preprocessor.joblib')

In [171]:
if create_and_save_artifacts:
    tabular_preprocessor = ColumnTransformer(
        transformers=[
            ('num', StandardScaler(), numeric_features),
        ],
        remainder='drop' # Garde les autres colonnes si besoin
    )
    tabular_preprocessor.fit(X_train)

    # Save
    path.parents[0].mkdir(parents=True, exist_ok=True)
    joblib.dump(tabular_preprocessor, path)

In [172]:
# Load
tabular_preprocessor = joblib.load(path)

In [173]:
X_train_tabular_processed = tabular_preprocessor.transform(X_train)
X_test_tabular_processed = tabular_preprocessor.transform(X_test)

#### Hashes

In [174]:
from sklearn.preprocessing import OrdinalEncoder
# OrdinalEncoder gère les données 2D (DataFrames)
# contrairement à LabelEncoder. handle_unknown est une bonne pratique pour l'inférence.

In [175]:
import joblib
path=artifacts_folder / 'hash_encoder.joblib'
path

PosixPath('artifacts/on_images/deep_learning/v1/hash_encoder.joblib')

In [176]:
if create_and_save_artifacts:
    hash_features=['image_pHash', 'image_hash_md5']
    hash_encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
    hash_encoder.fit(X_train[hash_features])

    X_train_processed_hashes = hash_encoder.transform(X_train[hash_features])

    # Save
    path.parents[0].mkdir(parents=True, exist_ok=True)
    joblib.dump(hash_encoder, path)

In [177]:
# Load
hash_encoder = joblib.load(path)

In [178]:
X_train_processed_hashes = hash_encoder.transform(X_train[hash_features])
X_test_processed_hashes = hash_encoder.transform(X_test[hash_features])

#### Smote (TODO)

In [179]:
#TODO: see Jessy's notebook
# Should this be earlier?

### Load tensorflow

In [180]:
import tensorflow as tf

In [181]:
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))

Num GPUs Available:  1


In [182]:
print(f"tensorflow: {tf.__version__}")

tensorflow: 2.20.0


#### tf dataset

In [183]:
# On prépare un dictionnaire de toutes nos entrées
train_inputs_dict = {
    'tabular_input': X_train_tabular_processed,
    'pHash_input': X_train_processed_hashes[:, 0],
    'md5_input': X_train_processed_hashes[:, 1],
    'image_input': X_train['image_path'].values
}
train_ds = tf.data.Dataset.from_tensor_slices((train_inputs_dict, y_train))

In [184]:
def load_image_and_format(inputs, label):
    # 'inputs' est le dictionnaire que nous venons de créer
    filepath = inputs['image_input']  # On récupère le chemin de l'image

    # On charge et on décode l'image
    image_raw = tf.io.read_file(filepath)
    image = tf.io.decode_jpeg(image_raw, channels=3)
    image = tf.image.resize(image, (500, 500))

    # On met à jour le dictionnaire : on remplace le chemin par le tenseur de l'image
    inputs['image_input'] = image

    return inputs, label

In [185]:
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

train_ds = (
    train_ds
    .shuffle(1000)
    .map(load_image_and_format, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(buffer_size=AUTOTUNE)  # prépare le prochain lot pendant que le GPU travaille sur le lot actuel
)

# TODO: exactement la même chose pour créer test_ds (sans le .shuffle())

## Model creation

### Architecture

In [186]:
from tensorflow import keras
from tensorflow.keras import layers

In [187]:
len(numeric_features)

24

#### Inputs

In [189]:
image_input = keras.Input(shape=(500, 500, 3), name="image_input")
tabular_input = keras.Input(shape=(len(numeric_features),), name='tabular_input')

In [190]:
pHash_vocab_size = len(hash_encoder.categories_[0])
md5_vocab_size = len(hash_encoder.categories_[1])
embedding_dim = 16  # La taille souhaitée du vecteur pour chaque hash. C'est un hyperparamètre.

In [191]:
pHash_input = keras.Input(shape=(1,), name='pHash_input', dtype='int64')  # Embedding a besoin du type int
md5_input = keras.Input(shape=(1,), name='md5_input', dtype='int64')

#### Image branch

In [192]:
# On utilise un modèle pré-entraîné.
# C'est une base très puissante pour traiter les images.
base_model = keras.applications.EfficientNetV2B0(
    include_top=False, # On ne garde que les couches d'extraction de features
    weights='imagenet', # Poids appris sur des millions d'images
    input_tensor=image_input
)
base_model.trainable = False # On "gèle" le modèle de base pour le début de l'entraînement

# On ajoute nos propres couches par-dessus
image_features = layers.GlobalAveragePooling2D(name='image_pooling')(base_model.output)
image_features = layers.Dense(128, activation='relu', name='image_dense')(image_features)

24274472/24274472 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step


#### Tabular branch

In [193]:
tabular_features = layers.Dense(64, activation='relu', name='tabular_dense_1')(tabular_input)
tabular_features = layers.Dense(32, activation='relu', name='tabular_dense_2')(tabular_features)

#### Hash branches

Chaque hash passe par sa propre couche d'Embedding.

In [ ]:
pHash_features = layers.Embedding(input_dim=pHash_vocab_size, output_dim=embedding_dim, name='pHash_embedding')(pHash_input)
pHash_features = layers.Flatten(name='pHash_flatten')(pHash_features)  # retire une dimension superflue de taille 1

md5_features = layers.Embedding(input_dim=md5_vocab_size, output_dim=embedding_dim, name='md5_embedding')(md5_input)
md5_features = layers.Flatten(name='md5_flatten')(md5_features)


#### Fusing branches


In [195]:
# On fusionne toutes les features apprises en un seul grand vecteur
all_features = layers.concatenate([
    image_features,
    tabular_features,
    pHash_features,
    md5_features
])

#### Classification

In [196]:
# Quelques couches denses pour apprendre les interactions entre les différentes modalités
x = layers.Dense(256, activation='relu', name='final_dense_1')(all_features)
x = layers.Dropout(0.5)(x)  # pour éviter l'overfitting
num_classes = 27  # nombre de classes
output = layers.Dense(num_classes, activation='softmax', name='output')(x)

#### Model

In [197]:
model = keras.Model(
    inputs=[image_input, tabular_input, pHash_input, md5_input],
    outputs=output
)

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ image_input         │ (None, 500, 500,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling           │ (None, 500, 500,  │          0 │ image_input[0][0] │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ normalization       │ (None, 500, 500,  │          0 │ rescaling[0][0]   │
│ (Normalization)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv (Conv2D)  │ (None, 250, 250,  │        864 │ normalization[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_bn             │ (None, 250, 250,  │        128 │ stem_conv[0][0]   │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_activation     │ (None, 250, 250,  │          0 │ stem_bn[0][0]     │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_project_co… │ (None, 250, 250,  │      4,608 │ stem_activation[… │
│ (Conv2D)            │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_project_bn  │ (None, 250, 250,  │         64 │ block1a_project_… │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_project_ac… │ (None, 250, 250,  │          0 │ block1a_project_… │
│ (Activation)        │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2a_expand_conv │ (None, 125, 125,  │      9,216 │ block1a_project_… │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2a_expand_bn   │ (None, 125, 125,  │        256 │ block2a_expand_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2a_expand_act… │ (None, 125, 125,  │          0 │ block2a_expand_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2a_project_co… │ (None, 125, 125,  │      2,048 │ block2a_expand_a… │
│ (Conv2D)            │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2a_project_bn  │ (None, 125, 125,  │        128 │ block2a_project_… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2b_expand_conv │ (None, 125, 125,  │     36,864 │ block2a_project_… │
│ (Conv2D)            │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2b_expand_bn   │ (None, 125, 125,  │        512 │ block2b_expand_c… │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2b_expand_act… │ (None, 125, 125,  │          0 │ block2b_expand_b

 Total params: 8,164,299 (31.14 MB)

 Trainable params: 2,244,987 (8.56 MB)

 Non-trainable params: 5,919,312 (22.58 MB)

## Training

### Saving and loading a model (TODO)

In [198]:
# # Callback pour sauvegarder le meilleur modèle au fur et à mesure
# save = ModelCheckpoint(
#     'best_model.h5',
#     save_best_only=True,
#     monitor='val_accuracy',
#     mode='max'
# )
# model_history = model.fit(train_ds,
#                           validation_data=val_ds,
#                           epochs=50,
#                           callbacks = [save])

In [199]:
# # ou sauvegarder l'intégralité du modèle après l'entraînement, c'est-à-dire son architecture et ses poids :
# model.save('mon_modele.keras')

In [200]:

# #    Pour charger un modèle sauvegardé on utilise la fonction load_model() de tensorflow.keras.models.
# model_loaded = load_model('best_model.keras')

### Training

In [201]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [202]:
# model.fit(train_ds, validation_data=test_ds, epochs=10)

## Evaluation